# P32 — Voyager: un agente encarnado de final abierto con modelos de lenguaje grandes

## 1. Título y paper

**Paper:** *Voyager: An Open-Ended Embodied Agent with Large Language Models*  
**Autoría:** Guanzhi Wang, Yuqi Xie, Yunfan Jiang, Ajay Mandlekar, Chaowei Xiao, Yuke Zhu, Linxi Fan, Anima Anandkumar  
**Año y venue:** 2023 · arXiv:2305.16291  
**Nivel:** L3 · **Motor:** `voyager`  
**Ficha completa:** [`P32_voyager`](../../papers/foundational/P32_voyager/README.md)

**Hito:** El agente acumula habilidades reutilizables en vez de contexto: memoria procedimental que no se borra al terminar la tarea.

- [arXiv:2305.16291](https://arxiv.org/abs/2305.16291)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un agente que resuelve tareas cada vez desde cero no mejora con la experiencia, y meter todo lo aprendido en el prompt no escala.
2. Ejecutar una implementación mínima de la propuesta: Un currículo automático que propone la siguiente tarea alcanzable, una biblioteca de habilidades ejecutables indexada por nombre, y un bucle iterativo que depura el código con la retroalimentación del entorno.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P13
- P30
- P14


## 4. Intuición

Aprender a cocinar no es recordar cada vez la receta entera: es que «hacer un sofrito» pase a ser una sola cosa que sabes hacer. Voyager guarda habilidades, no anécdotas.


## 5. Concepto mínimo

```text
Memoria en contexto:      cada tarea reintroduce todo lo aprendido como TEXTO
                          → ocupa contexto, se pierde al terminar

Biblioteca de habilidades: cada solución verificada se guarda como CÓDIGO con nombre
                          → se invoca por nombre, se compone, persiste
```

Más un **currículo automático**: el agente propone su siguiente tarea en función de lo que ya sabe y de lo que ve en el entorno.


## 6. Código explicado

El motor construye una biblioteca donde cada habilidad se apoya en las anteriores.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('voyager', seed=7)['result']
for c in r['curriculo']:
    print(f"{c['tarea']:<18} · {c['pasos_declarados']} pasos declarados "
          f"= {c['acciones_primitivas_equivalentes']:>2} primitivas "
          f"· reutiliza {c['habilidades_reutilizadas']}")

## 7. Predicción antes de ejecutar

1. ¿Cuántas acciones primitivas equivale la última tarea, declarada en 3 pasos?
2. ¿Qué pasaría si cada tarea tuviera que escribirse desde cero en primitivas?
3. ¿Por qué una biblioteca no consume contexto y una memoria textual sí?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
r = run_paper_lab('voyager', seed=7)['result']
print('habilidades en la biblioteca:', r['habilidades_aprendidas'], '\n')
for c in r['curriculo']:
    print(f"{c['tarea']:<18} factor de compresión {c['factor_de_compresion']:>5}x")

## 9. Salida interpretable

El factor de compresión crece con el currículo: las tareas tardías se expresan en muy pocos pasos porque cada uno esconde muchas primitivas. **Eso es lo que significa acumular capacidad**, frente a acumular texto.


## 10. Comentario pedagógico

La pieza que hace esto viable, y que la miniatura no muestra, es la **verificación**: una habilidad solo entra en la biblioteca si su código se ejecutó y funcionó en el entorno. Sin ese filtro, la biblioteca se llena de habilidades rotas que se propagan a todo lo que las use.


## 11. Error o anti-patrón deliberado

Anti-patrón: guardar en la biblioteca sin verificar. Una habilidad rota contamina todas las que la componen.


In [ ]:
biblioteca = {'conseguir_madera': 'ROTA (no comprueba si hay árbol)'}
dependen = ['fabricar_mesa', 'fabricar_pico', 'minar_piedra', 'fabricar_horno']
print('habilidad rota:', list(biblioteca)[0])
print('afectadas por composición:', dependen)
print(f'una sola habilidad sin verificar rompe {len(dependen)} tareas posteriores')

## 12. Corrección

La corrección es un contrato de entrada a la biblioteca:


In [ ]:
contrato = {
    'se_guarda_si': ['el código se ejecutó en el entorno',
                      'la tarea se completó de forma verificable',
                      'tiene nombre y descripción para poder recuperarla'],
    'se_reintenta_si': 'falla, con el error del entorno como retroalimentación',
    'nunca': 'guardar código que solo parece correcto',
}
show(contrato)

## 13. Desafío guiado

Comprueba cómo crece la capacidad al componer: cuenta las primitivas de una tarea inventada de nivel 6.


In [ ]:
r = run_paper_lab('voyager', seed=7)['result']
biblioteca = r['biblioteca_final']
print('biblioteca:')
for k, v in biblioteca.items():
    print(f'  {k:<18} = {v}')

## 14. Desafío autónomo

Construye un agente con biblioteca de habilidades para un entorno programable simple (por ejemplo un intérprete de comandos de ficheros). Exige verificación antes de guardar y mide cuántos pasos necesita para la tarea 10 con y sin biblioteca.


## 15. Evidencia de aprendizaje

Guarda el currículo con su factor de compresión, el experimento de la habilidad rota y tu contrato de entrada a la biblioteca.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P32_voyager/README.md) · evaluación formal: [`assessments/papers/P32_voyager.md`](../../assessments/papers/P32_voyager.md)


## 16. Cierre

Un agente que aprende y acumula. La última pregunta es si **varios** agentes se coordinan mejor que uno solo — y qué cuesta.


## 17. Conexión con el siguiente hito

- P16

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
